In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf

print("TensorFlow version:", tf.__version__)


In [ ]:

MODEL_PATH = Path("model/asl_1dcnn.keras")
DATA_PATH = Path("data/processed/asl_windows.npz")

OUT_TFLITE = Path("compression/asl_int8.tflite")
OUT_HEADER = Path("deployment/model.h")

REPRESENTATIVE_SAMPLES = 200

CLASS_NAMES = [
    "HELLO",
    "THANK_YOU",
    "HELP",
    "GOODBYE",
    "STOP",
    "MORE",
    "EAT",
    "WATER",
]

print("Model path:", MODEL_PATH)
print("Data path:", DATA_PATH)
print("Output TFLite:", OUT_TFLITE)
print("Output header:", OUT_HEADER)


In [ ]:
def load_windows(npz_path: Path):
    data = np.load(npz_path)

    if "X_train" not in data:
        raise KeyError(
            f"{npz_path} must contain X_train with shape "
            "(num_windows, window_size, 6)."
        )

    X_train = data["X_train"].astype(np.float32)

    X_eval = None
    y_eval = None

    for x_key, y_key in [("X_test", "y_test"), ("X_val", "y_val")]:
        if x_key in data and y_key in data:
            X_eval = data[x_key].astype(np.float32)
            y_eval = data[y_key].astype(np.int64)
            break

    if X_train.ndim != 3 or X_train.shape[-1] != 6:
        raise ValueError(
            f"Expected X_train shape (N, window_size, 6), got {X_train.shape}"
        )

    return X_train, X_eval, y_eval


X_train, X_eval, y_eval = load_windows(DATA_PATH)

print("X_train shape:", X_train.shape)
print("X_train dtype:", X_train.dtype)

if X_eval is not None:
    print("X_eval shape:", X_eval.shape)
    print("y_eval shape:", y_eval.shape)
else:
    print("No eval set found. Add X_test/y_test or X_val/y_val to evaluate accuracy.")


In [ ]:
def make_representative_dataset(X_train: np.ndarray, num_samples: int = 200):
    n = min(num_samples, len(X_train))
    if n == 0:
        raise ValueError("Representative dataset is empty.")

    # Deterministic sampling keeps conversion repeatable.
    indices = np.linspace(0, len(X_train) - 1, n, dtype=np.int64)

    def representative_dataset():
        for idx in indices:
            # TFLite expects a batch dimension.
            yield [X_train[idx : idx + 1].astype(np.float32)]

    return representative_dataset


rep_dataset = make_representative_dataset(X_train, REPRESENTATIVE_SAMPLES)
print(f"Representative samples: {min(REPRESENTATIVE_SAMPLES, len(X_train))}")


In [ ]:
def convert_to_full_int8(
    model_path: Path,
    X_train: np.ndarray,
    out_tflite: Path,
    representative_samples: int = 200,
):
    model = tf.keras.models.load_model(model_path)

    print("Loaded model:")
    model.summary()

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = make_representative_dataset(
        X_train, representative_samples
    )

    # Force fully integer inference for TensorFlow Lite Micro.
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()

    out_tflite.parent.mkdir(parents=True, exist_ok=True)
    out_tflite.write_bytes(tflite_model)

    print(f"Saved INT8 TFLite model to: {out_tflite}")
    print(f"Model size: {out_tflite.stat().st_size / 1024:.1f} KB")

    return tflite_model


tflite_model = convert_to_full_int8(
    model_path=MODEL_PATH,
    X_train=X_train,
    out_tflite=OUT_TFLITE,
    representative_samples=REPRESENTATIVE_SAMPLES,
)


In [ ]:
def inspect_tflite(tflite_path: Path):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    print("Input details")
    print("-------------")
    print("shape:", input_details["shape"])
    print("dtype:", input_details["dtype"])
    print("quantization:", input_details["quantization"])

    print("\nOutput details")
    print("--------------")
    print("shape:", output_details["shape"])
    print("dtype:", output_details["dtype"])
    print("quantization:", output_details["quantization"])

    return input_details, output_details


input_details, output_details = inspect_tflite(OUT_TFLITE)

INPUT_SCALE, INPUT_ZERO_POINT = input_details["quantization"]
OUTPUT_SCALE, OUTPUT_ZERO_POINT = output_details["quantization"]

print("\nUse these on Arduino if needed:")
print("INPUT_SCALE =", INPUT_SCALE)
print("INPUT_ZERO_POINT =", INPUT_ZERO_POINT)
print("OUTPUT_SCALE =", OUTPUT_SCALE)
print("OUTPUT_ZERO_POINT =", OUTPUT_ZERO_POINT)


In [ ]:
def quantize_float_to_int8(x_float: np.ndarray, scale: float, zero_point: int):
    if scale == 0:
        raise ValueError("Quantization scale is zero.")
    x_q = np.round(x_float / scale + zero_point)
    return np.clip(x_q, -128, 127).astype(np.int8)


def dequantize_int8(y_q: np.ndarray, scale: float, zero_point: int):
    return scale * (y_q.astype(np.float32) - zero_point)


def evaluate_int8(tflite_path: Path, X_eval: np.ndarray, y_eval: np.ndarray):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    in_scale, in_zero = input_details["quantization"]
    out_scale, out_zero = output_details["quantization"]

    correct = 0
    predictions = []

    for i in range(len(X_eval)):
        x_q = quantize_float_to_int8(X_eval[i : i + 1], in_scale, in_zero)

        interpreter.set_tensor(input_details["index"], x_q)
        interpreter.invoke()

        y_q = interpreter.get_tensor(output_details["index"])[0]
        y_float = dequantize_int8(y_q, out_scale, out_zero)

        pred = int(np.argmax(y_float))
        predictions.append(pred)
        correct += int(pred == int(y_eval[i]))

    accuracy = correct / len(y_eval)
    unique, counts = np.unique(predictions, return_counts=True)

    print(f"INT8 accuracy: {accuracy:.4f}")
    print("Prediction counts:")
    for cls_idx, count in zip(unique, counts):
        cls_name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f"class_{cls_idx}"
        print(f"  {cls_idx} ({cls_name}): {count}")

    return accuracy, np.array(predictions)


if X_eval is not None and y_eval is not None:
    int8_accuracy, int8_predictions = evaluate_int8(OUT_TFLITE, X_eval, y_eval)
else:
    print("Skipped eval because no X_test/y_test or X_val/y_val was found.")


In [ ]:
def predict_one_int8(tflite_path: Path, x_window: np.ndarray):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    in_scale, in_zero = input_details["quantization"]
    out_scale, out_zero = output_details["quantization"]

    x_q = quantize_float_to_int8(x_window[None, ...], in_scale, in_zero)

    interpreter.set_tensor(input_details["index"], x_q)
    interpreter.invoke()

    y_q = interpreter.get_tensor(output_details["index"])[0]
    y_float = dequantize_int8(y_q, out_scale, out_zero)

    pred_idx = int(np.argmax(y_float))
    pred_name = CLASS_NAMES[pred_idx] if pred_idx < len(CLASS_NAMES) else f"class_{pred_idx}"

    return pred_idx, pred_name, y_float


if X_eval is not None:
    for i in range(min(5, len(X_eval))):
        pred_idx, pred_name, scores = predict_one_int8(OUT_TFLITE, X_eval[i])
        actual = int(y_eval[i]) if y_eval is not None else None
        actual_name = CLASS_NAMES[actual] if actual is not None and actual < len(CLASS_NAMES) else actual

        print(f"Sample #{i+1}")
        print("  predicted:", pred_idx, pred_name)
        print("  actual:   ", actual, actual_name)
        print("  scores:   ", np.round(scores, 4))
else:
    print("No eval data available for sample predictions.")


In [ ]:
def write_c_header(tflite_path: Path, out_header: Path, array_name: str = "g_model"):
    model_bytes = tflite_path.read_bytes()
    out_header.parent.mkdir(parents=True, exist_ok=True)

    hex_values = [f"0x{b:02x}" for b in model_bytes]

    lines = []
    for i in range(0, len(hex_values), 12):
        lines.append("  " + ", ".join(hex_values[i : i + 12]) + ",")

    array_body = "\n".join(lines)

    header = f'''// Auto-generated from {tflite_path.name}
// INT8 ASL 1D-CNN model for TensorFlow Lite Micro.

#ifndef ASL_MODEL_H_
#define ASL_MODEL_H_

#include <cstdint>

alignas(8) const unsigned char {array_name}[] = {{
{array_body}
}};

const int {array_name}_len = {len(model_bytes)};

#endif  // ASL_MODEL_H_
'''

    out_header.write_text(header)

    print(f"Wrote header: {out_header}")
    print(f"Header model bytes: {len(model_bytes)}")


write_c_header(OUT_TFLITE, OUT_HEADER)


In [ ]:
print("Notebook complete.")
print("Generated files:")
print(" -", OUT_TFLITE)
print(" -", OUT_HEADER)
